# Time, place and features

Data Science & AI. Module 2 Part 1.

Monday covered the first two-thirds of EDA: is this data usable, what is
missing, what is invalid, what does one column look like, what do two
columns look like together.

Tonight is the third: the two column types that need their own toolkit
because *order* and *position* carry meaning.

- A **time series** is not a bag of numbers. Row 2 comes after row 1, and
  that fact is the information.
- **Geospatial** data is not two float columns. Latitude and longitude
  are useless apart and a map together.
- In between sits **feature engineering** — building the columns your
  model actually needs out of the columns your source actually gave you.

Everything runs. Nothing tonight is a slide you have to take on faith.

## Before you type anything

**Work on your own copy, not on this file.** File → Make a Copy, or
`cp` it into your `my-work/` folder. Then **Restart & Run All** once, so
you know it worked before we start breaking things on purpose.

## 0. Setup and the data

Tonight's dataset is the **UCI Bike Sharing** set: every hour that
Capital Bikeshare operated in Washington DC across 2011 and 2012, with
the weather at the time and how many bikes went out. 17,379 rows.

**One thing to know before you open the lab file.** IOD's lab uses a
*different cut of the same data* — the Kaggle "Bike Sharing Demand"
version, 10,886 rows, with columns called `datetime`, `weather`,
`humidity` and `count`. We are using the parent source from UCI: both
full years, and the columns are called `dteday` + `hr`, `weathersit`,
`hum` and `cnt`. Same bikes, same city, different packaging. If your lab
file's column names do not match tonight's, nothing is broken.

In [ ]:
# ==========================================================
# Setup. Run this once, then carry on.
# ==========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

UCI = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"


def load_bikes():
    """Local copy if it is beside us, otherwise download it from UCI."""
    try:
        return pd.read_csv("data/bikeshare-hour.csv")
    except FileNotFoundError:
        pass

    import io
    import urllib.request
    import zipfile

    print("no local copy — downloading from UCI ...")
    with urllib.request.urlopen(UCI, timeout=60) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))

    # The zip holds three files, so we have to name the one we want.
    with archive.open("hour.csv") as csv_file:
        return pd.read_csv(csv_file)


bikes = load_bikes()

pd.set_option("display.max_columns", 20)
print("rows:", len(bikes), " columns:", len(bikes.columns))
bikes.head(3)

If that `except` branch ran, notice what it did: went to a URL, pulled
down a zip file, and read a CSV out of it without ever writing anything
to your disk. That is slide 4's *"where does data come from"* in three
lines — a website is a data source like any other.

Now the columns. **Read the data dictionary before you read the data**
— UCI ships one, `bikeshare-README.txt`, and it says something that
`describe()` cannot tell you:

> **Predict first.** `temp` is temperature in Washington DC. `describe()` says its maximum is 1.0 and its mean is about 0.5. What is going on?
>
> Put your answer in the chat before we run it.

In [ ]:
print(bikes[["temp", "atemp", "hum", "windspeed"]].describe().round(3))

Every one of those four columns has been **normalised to 0-1** by whoever
built the file. The README gives the divisors: `temp` ÷ 41, `atemp` ÷ 50,
`hum` ÷ 100, `windspeed` ÷ 67.

So `temp = 0.5` is not half a degree and not 50%. It is **20.5°C**.

Nothing in the data would have told you. The summary statistics looked
perfectly reasonable and were perfectly meaningless. This is the whole
argument for reading documentation first:

In [ ]:
bikes["temp_c"] = bikes["temp"] * 41
bikes["hum_pct"] = bikes["hum"] * 100
bikes["wind_kmh"] = bikes["windspeed"] * 67

print("temperature in DC ranged from",
      round(bikes["temp_c"].min(), 1), "to",
      round(bikes["temp_c"].max(), 1), "degrees C")

That range is believable for Washington DC, which is the point — a
sanity check against the real world is a check the data cannot fake.

One more habit before we start. The file claims `cnt` is casual riders
plus registered riders. **Claims get asserted, not trusted:**

In [ ]:
assert (bikes["cnt"] == bikes["casual"] + bikes["registered"]).all()
print("cnt = casual + registered, on all", len(bikes), "rows")

---

## 1. What is a time series?  *(slides 52, 53)*

> **A time series is a sequence of data points representing the state of
> a system over time.**

The word doing the work is *sequence*. In every dataset you have met so
far, shuffling the rows changed nothing. Shuffle a time series and you
have destroyed it.

The deck splits them into two classes, and the split is worth keeping:

- **Temporally deterministic** — the future follows from the past by a
  rule. Sunrise times, a planet's position, the interest owed on a loan.
  You do not forecast these, you compute them.
- **Stochastic** — there is a rule *and* there is noise. Bike hires,
  share prices, hospital admissions, electricity demand. You forecast
  these, and you are always partly wrong.

Ours is firmly the second kind. And nearly every stochastic series in
the wild is three things added together:

| Component | In our data |
|---|---|
| **Trend** — the slow direction | Capital Bikeshare grew year on year |
| **Seasonality** — the repeating shape | Nobody cycles at 4am; nobody cycles in January |
| **Noise** — what is left | One wet Tuesday |

Most of tonight is learning to *see* those three separately.

## 2. Representing time in Python  *(slide 54)*

Our file stores time in two columns: `dteday`, a date, and `hr`, an hour
from 0 to 23. Pandas has read `dteday` as a **string**, because that is
all a CSV can offer:

In [ ]:
print(bikes["dteday"].dtype)
print(bikes["dteday"].head(3).tolist())

In [ ]:
bikes

`object` means "string". Before we do anything, watch what a string date
costs you. Here is a monthly total, plotted with the date left as text:

> **Predict first.** The x-axis will be months as text: '2011-01', '2011-02', ... '2012-12'. Matplotlib sorts them to draw them. Does the line come out in calendar order?
>
> Put your answer in the chat before we run it.

In [ ]:
text_months = bikes["dteday"].str[:7]              # '2011-01', '2011-02', ...
by_text = bikes.groupby(text_months)["cnt"].sum()

plt.figure(figsize=(10, 3))
plt.plot(by_text.index, by_text.values)
plt.xticks(rotation=90)
plt.title("Monthly hires, with the date left as a string")
plt.show()

That one is *nearly* fine, and that is exactly why it is dangerous.
`'2011-01' < '2011-02'` as text, so ISO-format dates happen to sort
correctly. Change the format and the illusion collapses:

> **Predict first.** Now the months are written the way most of the world writes them — '01/2011', '02/2011'. Sorted as text, where does January 2012 land?
>
> Put your answer in the chat before we run it.

In [ ]:
odd_months = bikes["dteday"].str[5:7] + "/" + bikes["dteday"].str[:4]
by_odd = bikes.groupby(odd_months)["cnt"].sum()

plt.figure(figsize=(10, 3))
plt.plot(by_odd.index, by_odd.values)
plt.xticks(rotation=90)
plt.title("The same data, dates as '01/2011' — this is now a lie")
plt.show()

January 2011 and January 2012 are now **neighbours**, because as text
they both start `01/`. The chart is smooth, labelled, professional, and
completely wrong. Nothing errored. Nobody would catch it from the code.

**Text is not time.** Convert it, always, first:

In [ ]:
bikes.head()

In [ ]:
# dteday gives the day; hr gives the hour. Add them to get the moment.
stamp = pd.to_datetime(bikes["dteday"]) + pd.to_timedelta(bikes["hr"], unit="h")

ts = bikes.set_index(stamp).sort_index()
ts.index.name = "when"
display(ts)
print(ts.index.dtype)
print(ts.index[:3])

`datetime64[ns]` is a real timestamp type, and a `DatetimeIndex` built
from it knows things a string never could:

In [ ]:
print("first hour in the file:", ts.index.min())
print("last hour in the file: ", ts.index.max())
print("that spans:", ts.index.max() - ts.index.min())
print()
print("the index knows its own parts:")
print("  year   ", ts.index.year)
print("  year   ", ts.index.year[0])
print("  month  ", ts.index.month[0])
print("  month  ", ts.index.month)
print("  weekday", ts.index.day_name()[0])

It also unlocks a piece of syntax you will use constantly — **slicing by
a partial date string**. Ask for a month, get a month:

In [ ]:
july = ts.loc["2012-07"]
display(july)
print("hours in July 2012:", len(july))
print("busiest single hour that month:", july["cnt"].max())

> **Predict first.** `resample` groups rows into time buckets. What do you think it needs, that `groupby` does not, before it can work?
>
> Put your answer in the chat before we run it.

In [ ]:
# Deliberate error. Read the LAST line of the traceback first.
bikes.resample("D")["cnt"].sum()

`TypeError: Only valid with DatetimeIndex...`

Read it bottom-up, as always. `resample` is not `groupby`. It does not
take a column to group by — it works on **the index**, and the index has
to be time. `bikes` still has the plain 0,1,2 index it was born with;
`ts` is the one we gave a `DatetimeIndex`.

Same call, right object:

In [ ]:
daily = ts["cnt"].resample("D").sum() #Regroup the rows into calendar-day buckets. "D" is daily; "h", "W" are the other common ones.
print(daily.head())
print("days:", len(daily))


## 3. Seeing a time series  *(slide 55)*

Once the index is time, plotting takes no arguments. Pandas reads the
index and labels the axis itself — that is the deck's *"default axis
labelling is aware of timebase"*:

In [ ]:
ts["cnt"].resample("D").sum()
ts["cnt"].resample("D").count()

In [ ]:
daily = ts["cnt"].resample("D").sum()
plt.figure(figsize=(11, 3))
daily.plot()
plt.title("Daily bike hires, 2011-2012")
plt.ylabel("hires")
plt.show()

Two of our three components are already visible: a **seasonal** hump
each summer, and a **trend** — the second summer is taller than the
first. The third component, noise, is the fuzz that makes the line thick.

To separate trend from noise, **resample into bigger buckets** or **roll a window**.
They are different tools and it is worth knowing which is which:

- `resample("W")` — chop time into weeks, one value per week. Fewer
  points out than in.
- `.rolling(30)` — for each day, average the 30 days up to it. Same
  number of points out as in, but smoothed.

In [ ]:
plt.figure(figsize=(11, 3))
daily.plot(alpha=0.3, label="daily")
daily.rolling(30).mean().plot(linewidth=2, label="30-day rolling mean")
#daily = ts["cnt"].resample("W").sum()
#daily.plot(alpha=0.3, label="per week")
plt.legend()
plt.title("Rolling means separate the trend from the noise")
plt.ylabel("hires")
plt.show()

The thin line is what happened. The thick line is what was *going on*.

Now put a number on the trend:

In [ ]:
by_year = ts.groupby(ts.index.year)["cnt"].sum()
print(by_year)
print()
growth = 100 * (by_year.iloc[1] / by_year.iloc[0] - 1)
print("year-on-year growth: %.0f%%" % growth)

### The daily shape, and the two populations hiding in it

Now the seasonality — but at the scale of a day rather than a year.
Average every 9am together, every 10am, and so on:

> **Predict first.** We are about to plot average hires against hour of day, 0 to 23. Sketch the shape you expect. One hump? Two? Where?
>
> Put your answer in the chat before we run it.

In [ ]:
ts.index.hour

In [ ]:
by_hour = ts.groupby(ts.index.hour)["cnt"].mean()

plt.figure(figsize=(9, 3))
by_hour.plot(marker="o")
plt.title("Average hires by hour of day — all days together")
plt.xlabel("hour")
plt.ylabel("mean hires")
plt.show()

Two spikes, morning and evening. That is a commute.

But "all days together" is doing something dishonest: it has averaged
Tuesdays and Sundays into one line, and those are **two different
populations of rider**. The file has a column that separates them —
`workingday`, 1 for a working day, 0 for a weekend or holiday. Split on
it:

In [ ]:
work = ts[ts["workingday"] == 1].groupby(ts[ts["workingday"] == 1].index.hour)["cnt"].mean()
rest = ts[ts["workingday"] == 0].groupby(ts[ts["workingday"] == 0].index.hour)["cnt"].mean()

plt.figure(figsize=(9, 3.5))
work.plot(marker="o", label="working days")
rest.plot(marker="o", label="weekends and holidays")
plt.legend()
plt.title("The same column, split — two completely different days")
plt.xlabel("hour")
plt.ylabel("mean hires")
plt.show()

Two different systems, sharing one dataset:

- Working days: sharp spikes at **8am** and **5-6pm**, a trough between.
  People riding to work.
- Weekends: one broad hump across the **middle of the day**. People
  riding for fun.

The combined chart showed neither honestly. It showed their average,
which describes nobody.


---

## 4. Feature engineering

Everything above came out of a single timestamp column. That is the
general move, and it has a name: **feature engineering** — building the
columns your analysis needs out of the columns your source gave you.

A `DatetimeIndex` is unusually generous here. One timestamp contains a
year, a month, a day, an hour, a weekday, and whether it is a weekend,
all waiting to be asked for:

In [ ]:
ts.index

In [ ]:
feat = pd.DataFrame(index=ts.index)
feat["hour"] = ts.index.hour
feat["weekday"] = ts.index.dayofweek          # Monday = 0
feat["month"] = ts.index.month
feat["is_weekend"] = ts.index.dayofweek >= 5
feat["is_rush"] = ts.index.hour.isin([7, 8, 17, 18])
feat["cnt"] = ts["cnt"]

print(feat.head(3))

### The trap in `hour`

`hour` looks like a number and behaves like one. Ask any model to use it
and it will assume 23 is far away from 0.

It is not. 23:00 and 00:00 are **one hour apart**. Midnight is next to
11pm and the column says it is eleven hours away — the whole calendar
wraps around and the plain integer does not.

The standard fix is to put the clock back on a circle, using the two
functions that *are* a circle:

In [ ]:
feat["hour_sin"] = np.sin(2 * np.pi * feat["hour"] / 24)
feat["hour_cos"] = np.cos(2 * np.pi * feat["hour"] / 24)

plt.figure(figsize=(4, 4))
plt.scatter(feat["hour_cos"], feat["hour_sin"], s=8)
for h in [0, 6, 12, 18]:
    plt.annotate(str(h),
                 (np.cos(2 * np.pi * h / 24), np.sin(2 * np.pi * h / 24)))
plt.axis("equal")
plt.title("24 hours, back on a clock face")
plt.show()

In [ ]:
feat

Every hour is now two numbers, and 23 sits right beside 0 where it
belongs. The same trick works for months, weekdays, and wind direction —
anything that wraps.

---

## 5. Geospatial data  *(slides 56-61)*

Latitude and longitude are two ordinary float columns that mean nothing
apart and everything together. Plot them as a scatter and you have
accidentally drawn a map.

### How the data is organised  *(slides 57, 58)*

| Format | What it is | Extension |
|---|---|---|
| **Shapefile** | The old ESRI GIS standard. Not one file — a set of them that must travel together. | `.shp` + `.dbf` + `.shx` |
| **KML / KMZ** | Keyhole Markup Language, built for Google Earth. XML underneath. | `.kml`, `.kmz` |
| **OpenStreetMap** | The crowdsourced map of the whole planet, free to use. | `.osm` |
| **GeoJSON** | JSON with a geometry field. The web standard, and the one you will meet most. | `.geojson` |

Underneath all of them are two ideas:

- **Points** — one place. A station, a shop, an incident.
- **Geometries** — lines and polygons. A road, a river, a suburb boundary.

And a **projection**, which is the answer to "the Earth is round and this
screen is not". `EPSG:4326` is plain latitude/longitude and is what you
will almost always get.

### The Python libraries  *(slides 60, 61)*

| Library | Does |
|---|---|
| **plotly** | Draws interactive maps. |
| **shapely** | Geometry maths — is this point inside that polygon, do these roads cross. |
| **fiona** | Reads and writes the GIS file formats above; handles projections. |
| **geopandas** | A DataFrame with a geometry column. Bundles all three. |


### Getting real coordinates, live

Our hourly counts have no location column — every row is the whole city.
But Capital Bikeshare publishes its stations in an open feed, no key and
no login, in a format called **GBFS**. Same bike system, same city.


In [ ]:
import requests

FEED = "https://gbfs.capitalbikeshare.com/gbfs/en/station_information.json"

response = requests.get(FEED, timeout=20)
response.raise_for_status()      # stops here, with a clear message, if the feed is down
feed = response.json()

stations = pd.DataFrame(feed["data"]["stations"])

print("stations:", len(stations))

print(stations[["name", "lat", "lon", "capacity"]].head(3))

In [ ]:
CATALOGUE = "https://raw.githubusercontent.com/MobilityData/gbfs/master/systems.csv"
systems = pd.read_csv(CATALOGUE)
hits = systems[systems["Location"].str.contains("NZ", case=False, na=False)]
hits

JSON in, DataFrame out, three lines. Note what just happened to the
shape of the problem: a nested structure from the internet became a
table, and everything you learned about tables now applies to it.

Profile it exactly as you would any other file:

In [ ]:
print(stations[["lat", "lon", "capacity"]].describe().round(3).T)

> **Predict first.** Latitude and longitude are just two float columns. Scatter one against the other. What will the picture look like?
>
> Put your answer in the chat before we run it.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(stations["lon"], stations["lat"], s=6, alpha=0.6)
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.title("Two float columns")
plt.axis("equal")
plt.show()

That is Washington DC — the dense core, the arms reaching out along the
Metro lines, the gap where the Potomac is. No map library involved. A
scatter plot of two numbers.

`plt.axis("equal")` is doing real work there. Without it, matplotlib
stretches the axes to fill the figure and the city comes out the wrong
shape. **A map with unequal axes is a distorted map.**

### A real map  *(slide 59)*


In [ ]:
import plotly.express as px

fig = px.scatter_map(
    stations, lat="lat", lon="lon", hover_name="name",
    zoom=10, height=600, map_style="open-street-map",
)
fig.update_traces(marker=dict(size=6, color="crimson"))
fig.show()

Scroll it. Zoom it. Click a station.

---

## 6. Business intelligence reporting  *(slides 63-66)*

Everything so far has been you, exploring, for you. **BI** is the other
job: collecting, analysing and visualising data so *someone else* can
decide something.

The difference is not the charts. It is the audience — and that they will
not be in the room when they read it.

Typical uses: financial reporting, operational alerts, inventory,
customer metrics. Typical tools: **Tableau**, **Microsoft Power BI**,
**Looker Studio**, **Qlik Sense**.

### The best-practice list  *(slide 65)*



1. **Ensure data validity.** Everything in sections 0 through 4 tonight
   happens *before* a dashboard exists. A polished dashboard built on
   unchecked data is worse than no dashboard, because now it is trusted.
2. **Consistent layout, relevant to the stakeholder.** Same shapes in the
   same places every time; they should not have to relearn it monthly.
3. **Label your axes, with units. Use legends and titles.** Our `temp`
   axis meant nothing until we knew it was ÷41.
4. **Smart interactivity** — filters and dropdowns, with sensible
   defaults and instructions. A dashboard opening on a blank state has
   failed already.
5. **Automate for reproducibility.** If refreshing it takes an afternoon
   of manual steps, it will silently go stale.
6. **Test it with real end users.** They will misread something you
   thought was obvious. Better to find out now.

### Tableau  *(slide 66)*

- Drag-and-drop, so a chart takes seconds and needs no code.
- Reads spreadsheets, CSVs, geospatial files, PDFs, databases, cloud
  storage.
- The **Data Source** tab is where you connect, clean and join — the
  same profiling work, in a GUI.
- **Sheet** → one chart. **Dashboard** → several sheets together.
  **Story** → dashboards in sequence, for presenting.
- **Show Me** suggests chart types for the columns you have selected.
- **Calculated fields** are new columns from formulas — feature
  engineering, in a GUI.


Gallery of what it can do: <https://www.tableau.com/viz-gallery>

**Lab 2.1.3 is the Tableau lab.** Tableau Public is free and enough for
the lab. Tableau Desktop is a 14-day trial — do not start that clock
until you actually need it.

---

## Your turn

Work in your own copy. Answers are checkable — each has an `assert` you
can run.

**1.** Build a *monthly* series of total hires with `resample`, and find
which calendar month across the two years was the busiest.

**2.** The `is_rush` feature used hours 7, 8, 17 and 18. Look back at the
working-day curve in section 3 and decide whether those are the right
four. Rebuild it with your own choice and compare the mean hires.

**3.** Average `temp_c` against mean hires per degree, bucketed with
`pd.cut` into 5°C bands. Is the relationship straight, or does it turn
over at the top end?


## Where to go deeper

- **Pandas time series user guide** —
  <https://pandas.pydata.org/docs/user_guide/timeseries.html>: the
  reference for `resample`, offsets and time zones. Dense, but it is the
  authority.
- **Forecasting: Principles and Practice**, Hyndman & Athanasopoulos —
  <https://otexts.com/fpp3/>: free, complete, and the best book on time
  series there is. Chapters 2 and 3 are tonight, done properly.
- **folium quickstart** —
  <https://python-visualization.github.io/folium/latest/getting_started.html>:
  markers, choropleths and layers, all short.
- **GBFS specification** —
  <https://github.com/MobilityData/gbfs>: the open standard we pulled
  from. Hundreds of cities publish it, so you can rebuild tonight's map
  for a city you actually live in.

---

*Data Science & AI — Session 11. Covers Module 2 Part 1 slides 52-67.
Official lab: IOD Lab 2.1.3 Tableau.*